# Aprendizado de Máquina — Lista prática 01

## Introdução ao Aprendizado de Máquina

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Na aula prática vimos que o erro de treino desce sempre e que o risco tem forma
de U. Aqui você vai refazer aquilo por conta própria, numa amostra nova, e a
lista termina numa pergunta que a Aula 01 não resolve:

> **se eu escolher o grau olhando o erro de teste de uma única amostra, quão
> confiável é essa escolha?**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula. As lacunas estão numeradas nos comentários, `# (a)`, `# (b)`, e
assim por diante.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
from matplotlib.pyplot import subplots

import sklearn.linear_model as skl
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

---
## Exercício 1 — a população

A população é a mesma da Aula 01:

$$X \sim \mathrm{Unif}[-3,3], \qquad
  Y = \underbrace{\sin(1{,}5X) + 0{,}3X}_{r(X)} + \varepsilon, \qquad
  \varepsilon \sim N(0;\,0{,}7^2).$$

Complete a função de regressão e o sorteio da amostra. Note que $r$ recebe um
vetor e devolve um vetor — nada de laço.

In [ ]:
A, B = -3.0, 3.0
SIGMA = 0.7


def r(x):
    """Função de regressão verdadeira, r(x) = E[Y | X = x]."""
    return ...      # (a)


def amostra(n, rng):
    """Sorteia n pares (x, y) da população."""
    x = ...                     # (b)
    y = r(x) + ...           # (c)
    return x, y

Confira o sorteio antes de seguir. O desvio-padrão de $Y$ tem de ficar bem acima
de $\sigma = 0{,}7$: além do ruído, $Y$ varia porque $r(X)$ varia.

In [ ]:
rng = np.random.default_rng(2026)
x_tr, y_tr = amostra(50, rng)
x_te, y_te = amostra(2000, rng)

print(f"treino: n = {len(x_tr)}, dp de y = {y_tr.std(ddof=1):.4f}")
print(f"teste:  n = {len(x_te)}")

> **Sua vez.** Desenhe a nuvem de treino e, por cima, a curva $r$ verdadeira numa
> grade fina de $x$. É a única vez no curso em que você pode fazer isso: numa
> população real, $r$ é justamente o que não se conhece.

---
## Exercício 2 — três graus, dois erros

Ajuste polinômios de grau 1, 5 e 15 à **mesma** amostra de treino e meça o erro
quadrático médio nos dois conjuntos.

O `Pipeline` abaixo já está montado: ele cria as potências de $x$, padroniza (para
o ajuste não sofrer com $x^{15}$ ser enorme) e resolve por mínimos quadrados.

In [ ]:
def modelo_poly(grau):
    return Pipeline([
        ("poly", PolynomialFeatures(degree=grau, include_bias=False)),
        ("escala", StandardScaler()),
        ("mqo", skl.LinearRegression()),
    ])


X_tr = x_tr.reshape(-1, 1)
X_te = x_te.reshape(-1, 1)

for grau in ...:                                     # (a)
    modelo = modelo_poly(grau).fit(..., ...)            # (b) ajuste no treino
    erro_tr = np.mean((y_tr - modelo.predict(X_tr)) ** 2)
    erro_te = ...   # (c) o mesmo, no teste
    print(f"grau {grau:2d}:  treino {erro_tr:.4f}   teste {erro_te:.4f}")

**Responda:** o grau 15 é o que menos erra no treino. Por que isso não é motivo
nenhum para escolhê-lo? Escreva a resposta na célula abaixo, como comentário.

---
## Exercício 3 — a curva em U

Agora varra os graus de 1 a 10, guarde os dois erros e desenhe as duas curvas.
Use escala logarítmica no eixo vertical: sem ela, o grau 10 achata todo o resto.

In [ ]:
graus = ...                                    # (a) array do numpy, de 1 a 10
erros_tr, erros_te = [], []

for grau in graus:
    modelo = modelo_poly(grau).fit(X_tr, y_tr)
    erros_tr.append(np.mean((y_tr - modelo.predict(X_tr)) ** 2))
    erros_te.append(np.mean((y_te - modelo.predict(X_te)) ** 2))

erros_tr, erros_te = np.array(erros_tr), np.array(erros_te)
melhor = graus[...]                    # (b) o grau de menor erro de teste
print(f"menor erro de teste: grau {melhor}  ({erros_te.min():.4f})")

In [ ]:
fig, ax = subplots(figsize=(5, 3.2))
ax.plot(graus, erros_tr, "s-", ms=4, label="erro de treino")
ax.plot(graus, erros_te, "o-", ms=4, label="erro de teste")
ax.axhline(..., ls="--", lw=1.2, color="gray",       # (c) o erro irredutível
           label=r"$\sigma^2$")
ax.set_yscale("log")
ax.set_xlabel("grau do polinômio")
ax.set_ylabel("erro quadrático médio")
ax.set_xticks(graus)
ax.legend()
fig.tight_layout()

---
## Exercício 4 — a escolha é confiável?

O Exercício 3 escolheu um grau a partir de **uma** amostra de treino. Repita o
experimento 200 vezes, com amostras de treino independentes, e olhe duas coisas
diferentes:

- o risco **médio** de cada grau — é o que a figura da nota mostra;
- em quantas das 200 amostras cada grau foi o vencedor.

Como conhecemos $r$, dá para calcular o risco sem sortear conjunto de teste:

$$R(\widehat r) = \mathbb{E}\big[(\widehat r(X) - r(X))^2\big] + \sigma^2,$$

bastando percorrer uma grade fina de $x$.

In [ ]:
rng = np.random.default_rng(2026)
x0 = np.linspace(A, B, 500)
r0 = r(x0)
X0 = x0.reshape(-1, 1)

riscos = np.zeros((200, len(graus)))
for b in range(200):
    x, y = amostra(50, rng)
    for j, grau in enumerate(graus):
        modelo = modelo_poly(grau).fit(x.reshape(-1, 1), y)
        riscos[b, j] = ...   # (a)

risco_medio = riscos.mean(axis=...)                           # (b) media sobre as amostras
print("melhor grau, em risco medio:", graus[int(np.argmin(risco_medio))])
for grau, v in zip(graus, risco_medio):
    print(f"  grau {grau:2d}: {v:.4f}")

Agora a segunda leitura: em cada uma das 200 amostras, qual grau venceu?

In [ ]:
vencedor = graus[...]                 # (a) o melhor grau de CADA amostra

for grau in graus:
    quantas = ...                 # (b)
    if quantas:
        print(f"grau {grau:2d}: venceu em {quantas:3d}/200  ({100 * quantas / 200:.1f}%)")

print(f"\ngrau 5 NAO foi o melhor em {100 * (vencedor != 5).mean():.1f}% das amostras")

> **Sua vez.** Repita a última célula com amostras de treino de $n=200$ em vez de
> $n=50$. A fração de amostras em que o grau 5 vence sobe ou desce? Era o que você
> esperava?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | o desvio-padrão de $Y$ (1,12) é bem maior que $\sigma$ (0,70): há sinal a estimar |
| 2 | o grau 15 tem o **menor** erro de treino e um erro de teste cinco ordens de grandeza maior |
| 3 | nesta amostra o melhor grau foi o **3**, não o 5 que a nota reporta |
| 4 | na média sobre 200 amostras o grau 5 ganha — mas perde em 31,5% delas |

**A seguir.** A Aula 02 traz a família de modelos que domina a prática quando $p$
é grande — regressão linear e suas versões regularizadas — e reencontra o mesmo
balanço por outro caminho. A ferramenta que resolve o problema do Exercício 4 vem
na Aula 03.